# 03 — Modelagem: Scorecard, Fairness & Calibração

Este notebook cobre a **Camada 2** do pipeline:

| Seção | O que demonstra |
|-------|-----------------|
| 1 · Carregamento de dados | Pipeline reproduzível via `params.yaml` |
| 2 · Baseline + métricas de crédito | KS / Gini / AUC — linguagem do mercado |
| 3 · Calibração (Platt scaling) | Probabilidades confiáveis para decisão de crédito |
| 4 · Análise de Fairness | Viés por gênero e faixa etária — LGPD / BACEN 4.557 |
| 5 · Scorecard (scorecardpy) | Pontuação interpretável para esteira de análise |
| 6 · MLflow tracking | Reprodutibilidade de experimentos |

> **Nota de reprodutibilidade:** para resultados consistentes, execute `make train` antes de rodar este notebook.
> O notebook assume que `data/processed/features.parquet` e `data/interim/german_credit.parquet` já existem.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from src.models.evaluate import (
    auc_roc,
    brier_score,
    credit_ks_statistic,
    expected_calibration_error,
    fairness_report,
    gini_coefficient,
)
from src.models.train import (
    _age_bins,
    _gender_proxy,
    build_base_pipeline,
    build_calibrated_pipeline,
)
from src.visualization.plots import (
    plot_fairness_auc,
    plot_reliability_diagram,
    plot_score_distribution_by_group,
)

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 110

PROJECT_ROOT = Path.cwd().parent
params = yaml.safe_load((PROJECT_ROOT / "params.yaml").read_text(encoding="utf-8"))
print("params carregados:", list(params.keys()))

## 1 · Carregamento de dados

In [ ]:
processed_path = PROJECT_ROOT / params["paths"]["processed_parquet"]
interim_path   = PROJECT_ROOT / params["paths"]["interim_parquet"]

df_feat    = pd.read_parquet(processed_path)
df_interim = pd.read_parquet(interim_path)

meta_path    = processed_path.parent / "feature_columns.json"
feature_cols = json.loads(meta_path.read_text(encoding="utf-8"))

X = df_feat[feature_cols]
y = df_feat["credit_risk"].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=params["split"]["test_size"],
    random_state=params["split"]["random_state"],
    stratify=y,
)

test_idx = X_test.index
print(f"Treino: {len(X_train)} | Teste: {len(X_test)} | Features: {len(feature_cols)}")
print(f"Taxa de bons (treino): {y_train.mean():.1%}")

---

## 2 · Baseline — Regressão Logística sem calibração

Métricas de mercado para crédito: **KS statistic**, **Gini** e **AUC-ROC**.

| Referência de mercado | KS | Gini |
|-----------------------|----|------|
| Insatisfatório | < 0.20 | < 0.30 |
| Aceitável | 0.20–0.40 | 0.30–0.50 |
| Bom | > 0.40 | > 0.50 |

In [ ]:
pipe_base = build_base_pipeline(feature_cols)
pipe_base.fit(X_train, y_train)
proba_base = pipe_base.predict_proba(X_test)[:, 1]

ks_base   = credit_ks_statistic(y_test, proba_base)
gini_base = gini_coefficient(y_test, proba_base)
auc_base  = auc_roc(y_test, proba_base)
bs_base   = brier_score(y_test, proba_base)
ece_base  = expected_calibration_error(y_test, proba_base)

print("─" * 40)
print(f"  KS statistic : {ks_base:.4f}")
print(f"  Gini         : {gini_base:.4f}")
print(f"  AUC-ROC      : {auc_base:.4f}")
print(f"  Brier score  : {bs_base:.4f}  (calibração — menor é melhor)")
print(f"  ECE          : {ece_base:.4f}  (calibração — menor é melhor)")
print("─" * 40)
print(classification_report(y_test, (proba_base >= 0.5).astype(int),
      target_names=["mau (0)", "bom (1)"]))

---

## 3 · Calibração de Probabilidades (Platt Scaling)

Em decisões de crédito, a **probabilidade precisa ser confiável** — não apenas o ranking.
Um score de 80% deve significar que ~80% dos tomadores naquela faixa realmente pagam.

**Platt scaling** (`CalibratedClassifierCV(method='sigmoid')`) ajusta a saída do classificador
com uma regressão logística adicional, minimizando o **Brier score** e o **ECE**.

Referência BACEN: Resolução 4.557/2017 exige que modelos internos tenham probabilidades calibradas.

In [ ]:
cal_method = params["calibration"]["method"]   # 'sigmoid' = Platt scaling
cal_cv     = params["calibration"]["cv"]

pipe_cal = build_calibrated_pipeline(feature_cols, method=cal_method, cv=cal_cv)
pipe_cal.fit(X_train, y_train)
proba_cal = pipe_cal.predict_proba(X_test)[:, 1]

bs_cal  = brier_score(y_test, proba_cal)
ece_cal = expected_calibration_error(y_test, proba_cal)

print(f"Brier score — sem calibração : {bs_base:.4f}")
print(f"Brier score — com Platt      : {bs_cal:.4f}   ({(bs_base - bs_cal) / bs_base:+.1%})")
print()
print(f"ECE — sem calibração         : {ece_base:.4f}")
print(f"ECE — com Platt              : {ece_cal:.4f}   ({(ece_base - ece_cal) / (ece_base + 1e-9):+.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Reliability diagrams — antes e depois da calibração
plot_reliability_diagram(y_test, proba_base, label="Sem calibração", color="#E24B4A", ax=axes[0])
axes[0].set_title(f"Sem calibração  (Brier={bs_base:.3f} | ECE={ece_base:.3f})")

plot_reliability_diagram(y_test, proba_cal, label="Platt scaling", color="#639922", ax=axes[1])
axes[1].set_title(f"Platt scaling   (Brier={bs_cal:.3f} | ECE={ece_cal:.3f})")

plt.suptitle("Reliability Diagram — impacto da calibração", fontsize=12)
plt.tight_layout()
plt.show()

---

## 4 · Análise de Fairness

Regulação brasileira (LGPD) e as diretrizes do BACEN exigem que modelos de crédito **não discriminem** por gênero ou faixa etária.

Métricas avaliadas:
- **Demographic Parity Difference** — diferença na taxa de aprovação entre grupos (idealmente próximo de 0)
- **Equal Opportunity Difference** — diferença na TPR (taxa de aprovação entre bons pagadores) (idealmente próximo de 0)
- **AUC por grupo** — poder discriminatório dentro de cada grupo

O atributo `personal_status_sex` do GCR codifica gênero + estado civil (1–5).
Proxy binário: códigos 1, 3, 4 → **male** | códigos 2, 5 → **female**.

In [ ]:
threshold = params["fairness"]["threshold"]

# --- Gênero ---
gender_test = _gender_proxy(df_interim.loc[test_idx, "personal_status_sex"])
fr_gender   = fairness_report(y_test, proba_cal, gender_test, threshold=threshold)

print("=== Fairness por gênero ===")
display_cols = ["n", "rate_positive", "selection_rate", "auc",
                "demographic_parity_diff", "equal_opportunity_diff"]
print(fr_gender[display_cols].round(4).to_string())

In [ ]:
# --- Faixa etária ---
age_bins   = params["fairness"]["age_bins"]
age_labels = params["fairness"]["age_labels"]
age_test   = _age_bins(df_interim.loc[test_idx, "age"], age_bins, age_labels)
fr_age     = fairness_report(y_test, proba_cal, age_test, threshold=threshold)

print("=== Fairness por faixa etária ===")
print(fr_age[display_cols].round(4).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

plot_fairness_auc(fr_gender, metric="auc",
                  title="AUC por gênero", ax=axes[0, 0])
plot_fairness_auc(fr_gender, metric="demographic_parity_diff",
                  title="Demographic Parity Diff — gênero", ax=axes[0, 1])
plot_fairness_auc(fr_age, metric="auc",
                  title="AUC por faixa etária", ax=axes[1, 0])
plot_fairness_auc(fr_age, metric="demographic_parity_diff",
                  title="Demographic Parity Diff — faixa etária", ax=axes[1, 1])

plt.suptitle("Análise de Fairness — LGPD / BACEN 4.557", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_score_distribution_by_group(
    proba_cal, gender_test, ax=axes[0])
axes[0].set_title("Distribuição de scores — gênero")

plot_score_distribution_by_group(
    proba_cal, age_test, ax=axes[1])
axes[1].set_title("Distribuição de scores — faixa etária")

plt.tight_layout()
plt.show()

### Interpretação dos resultados de fairness

- **Demographic Parity Diff ≈ 0**: grupos recebem aprovação na mesma proporção — **desejável** em contextos onde o regulador exige paridade de acesso ao crédito.
- **Demographic Parity Diff ≠ 0**: grupos com bases de risco distintas (inadimplência real diferente) podem ter `rate_positive` diferente — *não necessariamente viés*, mas requer documentação.
- **Equal Opportunity Diff**: diferença na taxa de aprovação *entre bons pagadores* — se alto, o modelo está sendo menos justo com bons pagadores de um grupo.
- **BACEN 4.557**: exige que a instituição documente e monitore esses indicadores em modelos de crédito.

---

## 5 · Scorecard Interpretável (scorecardpy)

O **scorecard** transforma o modelo em uma **tabela de pontos** que analistas de crédito conseguem auditar, explicar e aplicar manualmente — requisito comum em esteiras de crédito no Brasil.

A pontuação é calibrada para uma escala típica do mercado (ex.: 300–850, com 600 como ponto de corte de bom pagador).

> Requer o grupo Poetry `full` instalado: `poetry install --with full`

In [ ]:
try:
    import scorecardpy as sc
    SCORECARDPY_AVAILABLE = True
    print("scorecardpy disponível — versão:", sc.__version__)
except ImportError:
    SCORECARDPY_AVAILABLE = False
    print("scorecardpy não instalado. Execute: poetry install --with full")

In [ ]:
if SCORECARDPY_AVAILABLE:
    # scorecardpy opera sobre o DataFrame com os valores originais (não WoE)
    # Usamos o interim (pré-WoE) para o scorecard clássico
    df_sc = df_interim.copy()
    df_sc["credit_risk"] = df_feat["credit_risk"].astype(int)

    # Binning automático com IV mínimo de 0.02
    bins = sc.woebin(df_sc, y="credit_risk", min_iv=0.02)
    print(f"Features selecionadas pelo binning: {len(bins)}")

    # Converte para WoE e treina regressão logística
    df_sc_woe = sc.woebin_ply(df_sc, bins)
    X_sc = df_sc_woe.drop(columns=["credit_risk"])
    y_sc = df_sc_woe["credit_risk"]

    from sklearn.linear_model import LogisticRegression as LR
    lr_sc = LR(max_iter=500, class_weight="balanced", C=0.1)
    X_sc_train, X_sc_test, y_sc_train, y_sc_test = train_test_split(
        X_sc, y_sc, test_size=0.2, random_state=42, stratify=y_sc)
    lr_sc.fit(X_sc_train, y_sc_train)
    proba_sc = lr_sc.predict_proba(X_sc_test)[:, 1]

    print(f"KS scorecard   : {credit_ks_statistic(y_sc_test.values, proba_sc):.4f}")
    print(f"Gini scorecard : {gini_coefficient(y_sc_test.values, proba_sc):.4f}")
    print(f"AUC scorecard  : {auc_roc(y_sc_test.values, proba_sc):.4f}")

In [ ]:
if SCORECARDPY_AVAILABLE:
    # Geração do scorecard em pontos (escala 300–850, PDO=20, base=600)
    card = sc.scorecard(
        bins, lr_sc,
        X_sc_train.columns.tolist(),
        points0=600, odds0=1/19, pdo=20,
    )

    # Exibe as primeiras 3 features do scorecard
    for feat, tbl in list(card.items())[:3]:
        print(f"\n== {feat} ==")
        print(tbl.to_string(index=False))

In [ ]:
if SCORECARDPY_AVAILABLE:
    # Distribuição dos scores na escala de pontos
    sc_train = sc.scorecard_ply(df_sc.iloc[X_sc_train.index], card)
    sc_test  = sc.scorecard_ply(df_sc.iloc[X_sc_test.index],  card)

    fig, ax = plt.subplots(figsize=(9, 4))
    for label, scores, y_s, color in [
        ("mau (0)",  sc_test[sc_test.index.isin(y_sc_test[y_sc_test == 0].index)]["score"], None, "#E24B4A"),
        ("bom (1)",  sc_test[sc_test.index.isin(y_sc_test[y_sc_test == 1].index)]["score"], None, "#639922"),
    ]:
        ax.hist(scores, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.axvline(600, linestyle="--", color="black", label="corte 600")
    ax.set_xlabel("Score (pontos)")
    ax.set_ylabel("Densidade")
    ax.set_title("Distribuição do scorecard — bons vs maus")
    ax.legend()
    plt.tight_layout()
    plt.show()

---

## 6 · MLflow Tracking & Model Registry (Camada 3)

O `make train` executa o pipeline DVC completo e produz um run MLflow com:
- **Parâmetros:** modelo, método de calibração, CV, test_size, fingerprint do dataset
- **Métricas:** KS, Gini, AUC, Brier score, ECE + fairness por grupo
- **Assinatura do modelo:** `infer_signature` grava o schema input/output no Registry
- **Artefatos:** `reports/classification_report.txt`, 3 linhas de `input_example`

O ciclo de vida completo é gerenciado por `src/models/registry.py`:

```
treino → auto-registro (quality gate) → Staging → Production → Archived
```

Comandos do Makefile:
```bash
make mlflow                              # UI em http://localhost:5000
make evaluate                            # DVC evaluate → reports/metrics.json
make mlflow-register                     # registra o último run
make mlflow-list                         # lista versões no Registry
make mlflow-compare                      # compara runs (top-10 por KS)
make mlflow-promote-staging VERSION=2    # v2 → Staging
make mlflow-promote-prod    VERSION=2    # v2 → Production
```

```bash
make mlflow   # sobe a UI em http://localhost:5000
```

O `make train` executa o pipeline completo e loga:
- **Parâmetros:** modelo, método de calibração, CV, test_size, fingerprint do dataset
- **Métricas:** KS, Gini, AUC, Brier score, ECE
- **Métricas de fairness:** AUC / selection_rate / demographic_parity_diff por grupo
- **Artefatos:** `reports/classification_report.txt`, modelo joblib

In [ ]:
import mlflow

mlflow.set_tracking_uri("file://" + str(PROJECT_ROOT / "mlruns"))

from src.models.registry import compare_runs, get_latest_run_id, list_versions

# --- 6.1 Comparativo de runs (top-10 por KS) ---
print("=== Runs do experimento ===")
try:
    df_runs = compare_runs(
        params["mlflow"]["experiment_name"],
        top_n=10,
        sort_by="metrics.ks_statistic",
    )
    if not df_runs.empty:
        display_cols = [c for c in ["run_name", "p_model", "p_calibration_method",
                                    "ks_statistic", "gini", "auc_roc", "brier_score", "ece"]
                        if c in df_runs.columns]
        print(df_runs[display_cols].to_string(index=False))
    else:
        print("Nenhum run encontrado. Execute `make train` primeiro.")
except Exception as e:
    print(f"Erro ao consultar runs: {e}")

In [ ]:
# --- 6.2 Versões no Model Registry ---
print("=== Model Registry — versões registradas ===")
try:
    df_versions = list_versions(model_name=params["registry"]["model_name"])
    if not df_versions.empty:
        print(df_versions.to_string(index=False))
    else:
        print("Nenhuma versão registrada ainda.")
        print()
        print("Para registrar o último run:")
        print("  make mlflow-register")
        latest = get_latest_run_id(params["mlflow"]["experiment_name"])
        if latest:
            print(f"\nÚltimo run_id disponível: {latest}")
            print("Execute: make mlflow-register")
except Exception as e:
    print(f"Registry não acessível: {e}")

### Fluxo de promoção (Staging → Production)

```python
from src.models.registry import promote_to_staging, promote_to_production

# 1. Promove para Staging — passa pelo quality gate de params.yaml
promote_to_staging(version=1)

# 2. Promove para Production — arquiva a versão anterior automaticamente
promote_to_production(version=1, archive_existing=True)

# 3. Carrega o modelo Production para scoring
from src.models.predict import predict_proba
proba = predict_proba(X_test, use_registry=True, registry_stage="Production")
```

Quality gates definidos em `params.yaml`:
```yaml
registry:
  staging_threshold:
    ks_statistic: 0.25
    gini: 0.30
    auc_roc: 0.70
    brier_score: 0.25   # teto (menor = melhor)
```

---

## Resumo das Camadas 2 e 3

### Camada 2 — Modelagem

| Componente | Implementação | Arquivo |
|------------|---------------|---------|
| Calibração (Platt scaling) | `CalibratedClassifierCV(method='sigmoid')` | `src/models/train.py` |
| Brier score + ECE | `brier_score`, `expected_calibration_error` | `src/models/evaluate.py` |
| Fairness por gênero | `_gender_proxy` + `fairness_report` | `src/models/train.py` |
| Fairness por faixa etária | `_age_bins` + `fairness_report` | `src/models/train.py` |
| Scorecard (pontos) | `scorecardpy.scorecard` | notebook (grupo `full`) |

### Camada 3 — MLflow & Reprodutibilidade

| Componente | Implementação | Arquivo |
|------------|---------------|---------|
| Assinatura do modelo | `mlflow.models.infer_signature` | `src/models/train.py` |
| Auto-registro com quality gate | `_auto_register` + `_passes_threshold` | `src/models/train.py` |
| Ciclo de vida do Registry | `register_run`, `promote_to_staging`, `promote_to_production` | `src/models/registry.py` |
| Comparativo de runs | `compare_runs`, `list_versions` | `src/models/registry.py` |
| DVC metrics | `reports/metrics.json` (KS, Gini, AUC, Brier, ECE, fairness) | `src/models/evaluate_pipeline.py` |
| DVC plots | `reports/calibration_curve.csv`, `score_distribution.csv` | `src/models/evaluate_pipeline.py` |
| Load from Registry | `predict_proba(use_registry=True)` | `src/models/predict.py` |

**Próximo passo:** `04_report.ipynb` — relatório executivo completo com todas as seções obrigatórias pelo AGENTS.md.